In [25]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza z punktu 1.4.

client = bigquery.Client(credentials=credentials, project=credentials.project_id) 

In [26]:
gsod:pd.DataFrame
if False:
    query = """
    SELECT *
    FROM `bigquery-public-data.noaa_gsod.gsod2020`
    """

    query_job = client.query(query)
    query_result = query_job.result()
    gsod = query_result.to_dataframe()
    gsod.to_csv("gsod2020.csv.gz", index=False)
else:
    gsod = pd.read_csv("gsod2020.csv.gz")


C:\Users\Adrian\AppData\Local\Temp\ipykernel_18228\3958620364.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gsod = pd.read_csv("gsod2020.csv.gz")


In [27]:
gsod

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,...,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,010420,99999,2020-02-28,2020,2,28,23.2,4,9999.9,0,...,*,0.00,I,999.9,0,0,0,0,0,0
1,010980,99999,2020-09-30,2020,9,30,51.4,4,47.4,4,...,NaN,0.00,I,999.9,0,0,0,0,0,0
2,012120,99999,2020-10-04,2020,10,4,60.3,4,46.9,4,...,NaN,0.00,I,999.9,0,0,0,0,0,0
3,012620,99999,2020-03-13,2020,3,13,28.4,4,9999.9,0,...,NaN,0.00,I,999.9,0,0,0,0,0,0
4,013600,99999,2020-02-23,2020,2,23,29.1,4,22.7,4,...,NaN,0.24,G,10.6,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119200,A07056,464,2020-01-19,2020,1,19,21.5,24,13.7,24,...,*,0.00,I,999.9,0,0,0,0,0,0
4119201,A07086,468,2020-07-14,2020,7,14,72.9,24,60.9,24,...,*,0.00,I,999.9,0,0,0,0,0,0
4119202,A07357,182,2020-07-13,2020,7,13,73.3,24,64.3,24,...,*,0.00,I,999.9,0,0,0,0,0,0
4119203,A51255,445,2020-06-17,2020,6,17,71.5,24,62.1,24,...,*,0.00,I,999.9,0,0,0,0,0,0


In [28]:
isd = pd.read_csv("https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv")
isd.head()

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
0,007018,99999,WXPOD 7018,NaN,NaN,NaN,0.00,0.000,7018.0,20110309,20130730
1,007026,99999,WXPOD 7026,AF,NaN,NaN,0.00,0.000,7026.0,20120713,20170822
2,007070,99999,WXPOD 7070,AF,NaN,NaN,0.00,0.000,7070.0,20140923,20150926
3,008260,99999,WXPOD8270,NaN,NaN,NaN,0.00,0.000,0.0,20050101,20120731
4,008268,99999,WXPOD8278,AF,NaN,NaN,32.95,65.567,1156.7,20100519,20120323


In [29]:
isd["WBAN"] = isd["WBAN"].apply(lambda x: str(x).zfill(5))
gsod["wban"] = gsod["wban"].apply(lambda x: str(x).zfill(5))
df = gsod.merge(isd, how="left", left_on=["stn", "wban"], right_on=["USAF", "WBAN"])

Czyszczenie danych z nieprawidłowych wartości

In [30]:
df["temp"] = df["temp"].replace(9999.9, np.nan)
df["dewp"] = df["dewp"].replace(9999.9, np.nan)
df["slp"] = df["slp"].replace(9999.9, np.nan)
df["stp"] = df["stp"].replace(9999.9, np.nan) # podejrzana max wartość 999.9 (ale to blisko 1 bar)
df["visib"] = df["visib"].replace(999.9, np.nan)
df["wdsp"] = df["wdsp"].replace(999.9, np.nan)
df["mxpsd"] = df["mxpsd"].replace(999.9, np.nan)
df["gust"] = df["gust"].replace(999.9, np.nan)
df["max"] = df["max"].replace(9999.9, np.nan)
df["min"] = df["min"].replace(9999.9, np.nan)
df["prcp"] = df["prcp"].replace(99.99, np.nan)
df["sndp"] = df["sndp"].replace(999.9, np.nan)

In [31]:
df["fog"] = df["fog"].astype('bool')
df["rain_drizzle"] = df["rain_drizzle"].astype('bool')
df["snow_ice_pellets"] = df["snow_ice_pellets"].astype('bool')
df["hail"] = df["hail"].astype('bool')
df["thunder"] = df["thunder"].astype('bool')
df["tornado_funnel_cloud"] = df["tornado_funnel_cloud"].astype('bool')

df["date"] = pd.to_datetime(df["date"])

In [32]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,4119205,2020-06-30 16:34:45.543401472,2020-01-01 00:00:00,2020-03-30 00:00:00,2020-06-30 00:00:00,2020-10-01 00:00:00,2020-12-31 00:00:00,NaN
year,4119205.0,2020.0,2020.0,2020.0,2020.0,2020.0,2020.0,0.0
mo,4119205.0,6.487342,1.0,3.0,6.0,10.0,12.0,3.463681
da,4119205.0,15.748153,1.0,8.0,16.0,23.0,31.0,8.809447
temp,4119205.0,55.55154,-105.0,41.7,57.8,73.3,110.0,22.468503
count_temp,4119205.0,18.173296,4.0,8.0,24.0,24.0,24.0,7.416018
dewp,3916855.0,44.311402,-112.6,31.3,45.7,59.8,89.9,21.171256
count_dewp,4119205.0,17.080642,0.0,8.0,23.0,24.0,24.0,8.238555
slp,2705716.0,1014.403294,933.5,1009.3,1014.1,1019.8,1076.6,9.412161
count_slp,4119205.0,10.347779,0.0,0.0,8.0,23.0,24.0,9.762281


3.1 Ilość wierszy z danymi

In [33]:
len(df)

4119205

3.2 Ilość krajów

In [34]:
df["CTRY"].unique()

array(['NO', 'SP', 'PO', 'PL', 'MK', 'BK', 'TU', 'RS', 'BO', 'KZ', 'AM',
       'SY', 'IZ', 'IR', 'PK', 'IN', 'CE', 'BM', 'MY', 'VM', 'LA', 'CH',
       'MO', 'AG', 'TS', 'PU', 'GV', 'SL', 'LY', 'ET', 'KE', 'UG', 'CG',
       'RW', 'CD', 'TO', 'GH', 'MA', 'ZI', 'BC', 'SF', 'ZA', 'US', 'CA',
       'MX', 'AV', 'CO', 'GY', 'BR', 'EC', 'BL', 'CI', 'PA', 'AR', 'AY',
       'AQ', 'FM', 'RM', 'BP', 'NH', 'KR', 'WS', 'TN', 'AU', 'AS', 'ID',
       'RP', 'IC', 'GM', 'BU', 'GR', 'KU', 'QA', 'BG', 'MG', 'NP', 'TH',
       'SH', 'EU', 'TE', 'CF', 'NI', 'UV', 'AO', 'MZ', 'GT', 'FR', 'LG',
       'LH', 'UP', 'KG', 'GG', 'AJ', 'UZ', 'TI', 'TX', 'IS', 'JO', 'SA',
       'AF', 'MV', 'KS', 'NG', 'ML', 'GB', 'IV', 'WA', 'DR', 'UY', 'PH',
       nan, 'UK', 'RI', 'AL', 'IT', 'CV', 'BT', 'JA', 'BY', 'WZ', 'VE',
       'PE', 'SW', 'MU', 'TP', 'EG', 'CT', 'EK', 'HO', 'PM', 'NS', 'HR',
       'MP', 'BN', 'FK', 'FP', 'GL', 'DA', 'SV', 'LE', 'FS', 'GA', 'CM',
       'CW', 'NZ', 'MD', 'PP', 'BE', 'TR', 'CY', 'AI

In [35]:
len(df["CTRY"].unique())

263

3.3. W jaki sposób zapisywane są dzienne informacje dla poszczególnych lokalizacji.

In [36]:
df[["stn", "wban", "date", "year", "mo", "da"]].head()

,stn,wban,date,year,mo,da
0,010420,99999,2020-02-28,2020,2,28
1,010980,99999,2020-09-30,2020,9,30
2,012120,99999,2020-10-04,2020,10,4
3,012620,99999,2020-03-13,2020,3,13
4,013600,99999,2020-02-23,2020,2,23


3.4. Jak zapisane są wartości liczbowe

Chyba stopnie Fahrenheita

In [37]:
df["temp"]

0          23.2
1          51.4
2          60.3
3          28.4
4          29.1
           ... 
4119200    21.5
4119201    72.9
4119202    73.3
4119203    71.5
4119204    75.7
Name: temp, Length: 4119205, dtype: float64

3.5 Przedział czasowy

In [38]:
df["date"].describe().T

count                          4119205
mean     2020-06-30 16:34:45.543401472
min                2020-01-01 00:00:00
25%                2020-03-30 00:00:00
50%                2020-06-30 00:00:00
75%                2020-10-01 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

In [39]:
df[(df["prcp"].notna()) & (df["date"] != 0)]["date"].describe().T

count                          3813230
mean     2020-06-30 21:48:30.680761344
min                2020-01-01 00:00:00
25%                2020-03-31 00:00:00
50%                2020-06-30 00:00:00
75%                2020-09-30 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

In [40]:
df[(df["temp"].notna())]["date"].describe().T

count                          4119205
mean     2020-06-30 16:34:45.543401472
min                2020-01-01 00:00:00
25%                2020-03-30 00:00:00
50%                2020-06-30 00:00:00
75%                2020-10-01 00:00:00
max                2020-12-31 00:00:00
Name: date, dtype: object

3.6 Więcej informacji o danych pogodowych.

In [54]:
df[["dewp", "temp", "fog", "wdsp", "snow_ice_pellets"]]

,dewp,temp,fog,wdsp,snow_ice_pellets
0,NaN,23.2,False,10.3,False
1,47.4,51.4,False,3.5,False
2,46.9,60.3,False,18.0,False
3,NaN,28.4,False,28.2,False
4,22.7,29.1,False,8.7,True
...,...,...,...,...,...
4119200,13.7,21.5,False,10.6,False
4119201,60.9,72.9,False,3.5,False
4119202,64.3,73.3,False,12.5,False
4119203,62.1,71.5,False,0.7,False


4.1. Podstawowe informacje o lokalizacjach pomiarów pogodowych (stacje) oraz krajach, tak aby dane były zrozumiałe dla człowieka i możliwe do dalszego przetwarzania.

In [42]:
country_codes = pd.read_csv("https://raw.githubusercontent.com/mysociety/gaze/refs/heads/master/data/fips-10-4-to-iso-country-codes.csv")
country_codes

,FIPS 10-4,ISO 3166,Name
0,AF,AF,Afghanistan
1,AX,-,Akrotiri
2,AL,AL,Albania
3,AG,DZ,Algeria
4,AQ,AS,American Samoa
...,...,...,...
274,-,-,World
275,YM,YE,Yemen
276,-,-,Zaire
277,ZA,ZM,Zambia


In [43]:
stations = df[["stn", "wban", "STATION NAME", "CTRY", "STATE", "LAT", "LON"]].drop_duplicates()
stations = stations[stations["CTRY"].notna()]

stations_merged = stations.merge(
    country_codes, 
    how="left", 
    left_on="CTRY", 
    right_on="FIPS 10-4"
)
stations_merged.rename(columns={"Name": "Country", "STATION NAME": "Station Name", "STATE": "State", "LAT": "lat", "LON": "lon"}, inplace=True)
stations_merged[["stn", "wban", "Station Name", "Country", "State", "lat", "lon", "FIPS 10-4", "ISO 3166"]]
stations_merged.to_csv("stations.csv", index=False)

4.2. Chcemy wygenerować podstawowe zestawienia dotyczące warunków pogodowych na świecie (np. temperatura, opady, wiatr) w ujęciu dziennym.

In [44]:
simplified_df = df.sort_values("date")[["date", "stn", "wban", "temp", "prcp", "wdsp"]]

In [45]:
simplified_df.to_csv("weather.csv.gz", index=False)

4.3. Chcemy poznać zjawiska ekstremalne w danych pogodowych poprzez uwypuklenie skrajnych wartości (np. bardzo wysokie/niskie temperatury, intensywne opady, silny wiatr).

In [46]:
temp_low, temp_high = simplified_df['temp'].quantile([0.01, 0.99])
prcp_high = simplified_df['prcp'].quantile(0.99)
wdsp_high = simplified_df['wdsp'].quantile(0.99)

# filtrowanie ekstremów
extreme_records = simplified_df[
    (simplified_df['temp'] < temp_low) | (simplified_df['temp'] > temp_high) |
    (simplified_df['prcp'] > prcp_high) |
    (simplified_df['wdsp'] > wdsp_high)
]
extreme_records.to_csv("extreme_weather.csv.gz", index=False)

In [47]:
extreme_records

,date,stn,wban,temp,prcp,wdsp
1339718,2020-01-01,030020,99999,47.4,0.01,22.4
2207272,2020-01-01,119160,99999,16.3,0.00,30.8
229410,2020-01-01,967510,99999,66.8,2.82,1.5
2413215,2020-01-01,702960,26410,33.7,2.53,7.4
3364580,2020-01-01,726690,24057,23.7,0.06,24.3
...,...,...,...,...,...,...
1097627,2020-12-31,306270,99999,-26.7,0.00,5.1
958538,2020-12-31,716290,99999,-16.0,0.02,0.0
788705,2020-12-31,251210,99999,-34.7,0.00,6.1
3583864,2020-12-31,999999,26564,-24.6,0.00,NaN


4.4. Chcemy przygotować dane umożliwiające analizę zmian warunków pogodowych w czasie (np. dla wybranych lokalizacji lub zmiennych).